# 01. Сбор данных с krisha.kz — Астана

Собираем объявления о **продаже** и **аренде** квартир в Астане.

| Шаг | Что делает | Время |
|---|---|---|
| 1 | Установка библиотек | 1 мин |
| 2 | Пробный запуск: 2 страницы продажи + 2 аренды | 1 мин |
| 3 | Проверка результата | — |
| 4 | Все списки объявлений (цена, комнаты, площадь, этаж, адрес) | ≈30–60 мин |
| 5 | Детали каждого объявления (год постройки, тип дома, состояние, координаты) | несколько часов, можно прерывать |

> Файл `krisha_parser.py` должен лежать в той же папке, что и этот ноутбук.

In [ ]:
# Шаг 1. Установка библиотек (один раз)
%pip install requests beautifulsoup4 lxml pandas

In [ ]:
# Шаг 2. Пробный запуск
!python -u krisha_parser.py --cities astana --pages 2

In [ ]:
# Шаг 3. Проверяем результат
import glob
import pandas as pd

files = sorted(glob.glob("data/raw/listings_*.csv"))
df = pd.read_csv(files[-1])
print(f"Файл: {files[-1]} | строк: {len(df)}")
print(df.groupby("deal").size())
print("\nПропуски по колонкам:")
print(df.isna().sum())
df.sample(min(10, len(df)), random_state=1)

**Проверь:** у большинства строк заполнены `price_kzt`, `rooms`, `area_m2`, `floor`, `address`.
Если что-то не так или парсер написал «карточки не найдены» — пришли скриншот или файл из `data/debug/`.

Если всё в порядке — переходи к шагу 4.

In [ ]:
# Шаг 4. Все списки объявлений (≈20 объявлений на странице)
!python -u krisha_parser.py --cities astana --pages 150

### Шаг 5. Детали объявлений

Ноутбук заходит в каждое объявление. Это долго, но процесс можно **прерывать и запускать снова**
(Kernel → Interrupt) — уже скачанные объявления пропускаются. Удобно запустить на ночь.

Сначала проверь на 10 объявлениях:

In [ ]:
!python -u krisha_parser.py --details-only --details-limit 10

In [ ]:
details = pd.read_json("data/raw/details.jsonl", lines=True)
print(f"Скачано деталей: {len(details)}")
details.head(10)

Если поля (год постройки, тип дома и т.д.) заполнены — запускай полную загрузку:

In [ ]:
!python -u krisha_parser.py --details-only